In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import random
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
# Data Import
dataset_og = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")

# Pre-processing Data

In [3]:
# Copy the dataframe without 'Sex'
dataset_op = dataset_og[
    ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
].copy()

# 'Sex' feature needs masking
dataset_op['Sex'] = dataset_og['Sex'].map({'male': 1, 'female': 0})

# fillna U for unregistered, as most passengers didn't have cabins
deck_series = dataset_og['Cabin'].fillna('U').astype(str).str[0]

# What deck the cabin was on should impact survivability
# pd.get_dummies() is a Pandas function for one-hot encoding
deck_dummies = pd.get_dummies(deck_series, prefix='Cabin')
dataset_op = pd.concat([dataset_op, deck_dummies], axis=1)

# This matters because traveling with children often gave passengers priority on certain lifeboats
# The +1 is for the passenger himself. So, 1 means he is a solo traveler
dataset_op['TravelGroupSize'] = dataset_og['SibSp'] + dataset_og['Parch'] + 1

# Titles can be an important feature.
# * Mr. is always male
# * Ms. or Mrs. could tell marital status 
# * Title like Dr., Sir, Madam etc shows socioeconomic status etc
titles = dataset_og['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
# We shall filter and replace rare titles to handle over fitting
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
titles = titles.replace({
    'Mlle': 'Miss',
    'Ms': 'Miss',
    'Mme': 'Mrs'
})
titles = titles.replace(rare_titles, 'Rare')
titles_dummies = pd.get_dummies(titles, prefix='Titles_')
dataset_op = pd.concat([dataset_op, titles_dummies], axis=1)

# This is important feature as not all ports may have been visited by the time Titanic sank
embarked_ports = pd.get_dummies(dataset_og['Embarked'], prefix='Embarked_')
dataset_op = pd.concat([dataset_op, embarked_ports], axis=1)

# Fare is right skewed, so we are using log
dataset_op['Fare'] = np.log1p(dataset_op['Fare'])

# Create target column
dataset_op['Result'] = dataset_og['Survived']

# Test Train Split

In [4]:
# 1. Separate features and target
X = dataset_op.drop(columns=['Result'])
y = dataset_op['Result']

# 2. Split into train and validation sets first (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# KNN Imputation on missing Age
### Impute test and train seperately so test and train do not affect each other

In [5]:
# 1. Initialize KNN Imputer (n_neighbors=5 is a common default)
imputer = KNNImputer(n_neighbors=5)

# 4. Fit ONLY on X_train, then transform both X_train and X_val
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

# Random Forest

In [6]:
# Initialize model
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

ensemble = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
                    n_estimators=200,
                    max_depth=None,
                    min_samples_split=2,
                    min_samples_leaf=1,
                    random_state=42
                )),
        ('gb', HistGradientBoostingClassifier(random_state=42)),
        ('lr', LogisticRegression(max_iter=1000, random_state=42))
    ],
    voting='soft'
)

# Train
ensemble.fit(X_train, y_train)

# Predict
y_pred = ensemble.predict(X_test)

# Evaluate (Fixed y_val to y_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Validation Accuracy: {accuracy * 100:.4f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Validation Accuracy: 83.7989%

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.87      0.86       105
           1       0.81      0.80      0.80        74

    accuracy                           0.84       179
   macro avg       0.83      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179


Confusion Matrix:
[[91 14]
 [15 59]]
